In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
pd.set_option('display.max_columns', None)

In [ ]:
!pip install requests pandas pyarrow -q
# import orcid_enrichment

In [ ]:
#!/usr/bin/env python3
"""
orcid_enrichment.py  —  Google Colab
====================================
Harvest each researcher's DATED affiliation timeline from the public ORCID API,
then derive reusable per-author features. The raw timeline is the asset: it
answers many questions beyond Paper 3, so we STORE IT RAW and derive on top.

QUESTIONS THIS UNLOCKS
  - Seniority / career stage: years since first known affiliation (a proxy for
    academic age).
  - Cleaner author country: current/most-recent employment country, independent
    of a single paper's affiliation string.
  - Mobility: how many distinct countries/institutions in a career; migration
    sequence (e.g. left an African institution -> now Northern).
  - "Collaboration vs prior belonging": among authors who publish ABOUT Africa
    while affiliated in the North, did they PREVIOUSLY work at an African
    institution? (diaspora / returnee signal — a major equity nuance, since
    affiliation != origin.)
  - Reusable for future studies: gender-neutral career trajectories, brain-drain/
    brain-circulation, institution-to-institution flows, etc.

DESIGN
  Input = the unique ORCIDs in au_orcid_vec. The public API (no key) returns
  /record with employments + educations, each carrying start/end year + org name
  + org country (+ROR/GRID when present). We cache the RAW parsed timeline per
  ORCID (resumable), then build features. Re-run the feature step any time
  without re-hitting the API.

SETUP
  !pip install requests pandas pyarrow -q
  # ORCID public API is keyless for public data. A member token raises limits
  # (optional). Be polite: modest workers + backoff.
"""

import json, sys, time
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import pandas as pd, requests

# =========================================================================== #
# CONFIG                                                                        #
# =========================================================================== #
IN_PARQUET   = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/ror_enrich.parquet"
OUT_PARQUET  = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/with_orcid.parquet"
ORCID_CACHE  = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/orcid_enrich/orcid_timeline.jsonl"   # raw asset, keep!
MAILTO       = "cognosxx@gmail.com"
WORKERS      = 6

# Path(ORCID_CACHE).parent.mkdir(parents=True, exist_ok=True)


MAGHREB = {"MA","DZ","TN","LY","EG","MR"}
SSA = {"AO","BJ","BW","BF","BI","CV","CM","CF","TD","KM","CG","CD","CI","GQ","ER",
       "SZ","ET","GA","GM","GH","GN","GW","KE","LS","LR","MG","MW","ML","MZ","NA",
       "NE","NG","RW","ST","SN","SC","SL","SO","ZA","SS","SD","TZ","TG","UG","ZM","ZW"}
AFRICA = SSA | MAGHREB


def S(x):
    return "" if x is None or (isinstance(x, float) and pd.isna(x)) or x is pd.NA else str(x)

def clean_orcid(x):
    x = S(x).rsplit("/", 1)[-1].strip()
    return x if len(x) == 19 and x.count("-") == 3 else ""   # 0000-0000-0000-0000


# =========================================================================== #
# 1. collect unique ORCIDs from au_orcid_vec                                    #
# =========================================================================== #
def all_orcids(df):
    ids = set()
    for v in df["au_orcid_vec"].dropna():
        for slot in S(v).split("|"):        # '|' between authors, blank if none
            o = clean_orcid(slot)
            if o:
                ids.add(o)
    print(f"[orcid] {len(ids):,} unique ORCIDs", file=sys.stderr)
    return ids


# =========================================================================== #
# 2. harvest raw dated timeline (resumable)                                     #
# =========================================================================== #
def load_cache():
    done = {}
    if Path(ORCID_CACHE).exists():
        for line in open(ORCID_CACHE):
            try:
                d = json.loads(line); done[d["orcid"]] = d
            except Exception:
                pass
    print(f"[orcid] {len(done):,} cached", file=sys.stderr)
    return done

def _year(date_obj):
    try:
        return int(date_obj["year"]["value"])
    except (TypeError, KeyError, ValueError):
        return None

def _affiliations(group_json, kind):
    out = []
    for g in (group_json or {}).get("affiliation-group", []):
        for s in g.get("summaries", []):
            rec = s.get(f"{kind}-summary", {})
            org = rec.get("organization", {}) or {}
            addr = org.get("address", {}) or {}
            ids = (org.get("disambiguated-organization") or {})
            out.append({
                "kind": kind,
                "org": org.get("name", ""),
                "country": addr.get("country", "") or "",
                "role": rec.get("role-title", "") or "",
                "start": _year(rec.get("start-date")),
                "end": _year(rec.get("end-date")),
                "src_id": ids.get("disambiguated-organization-identifier", ""),
                "src": ids.get("disambiguation-source", ""),
            })
    return out

def _fetch(orcid, sess):
    url = f"https://pub.orcid.org/v3.0/{orcid}/record"
    for attempt in range(4):
        try:
            r = sess.get(url, headers={"Accept": "application/json"}, timeout=30)
            if r.status_code == 200:
                j = r.json()
                act = (j.get("activities-summary") or {})
                emp = _affiliations(act.get("employments"), "employment")
                edu = _affiliations(act.get("educations"), "education")
                return {"orcid": orcid, "affiliations": emp + edu}
            if r.status_code in (404, 409):
                return {"orcid": orcid, "affiliations": [], "note": str(r.status_code)}
            if r.status_code in (429, 500, 502, 503):
                time.sleep(2 ** attempt); continue
            return {"orcid": orcid, "affiliations": [], "note": str(r.status_code)}
        except requests.RequestException:
            time.sleep(2 ** attempt)
    return {"orcid": orcid, "affiliations": [], "note": "failed"}

def harvest(orcids):
    cache = load_cache()
    todo = [o for o in orcids if o not in cache]
    print(f"[orcid] {len(todo):,} to fetch", file=sys.stderr)
    sess = requests.Session()
    sess.headers.update({"User-Agent": f"tick-sos-study/1.0 (mailto:{MAILTO})"})
    fh = open(ORCID_CACHE, "a")
    with ThreadPoolExecutor(max_workers=WORKERS) as ex:
        futs = [ex.submit(_fetch, o, sess) for o in todo]
        for n, fut in enumerate(as_completed(futs), 1):
            d = fut.result()
            fh.write(json.dumps(d) + "\n"); fh.flush()
            cache[d["orcid"]] = d
            if n % 500 == 0:
                print(f"[orcid] {n:,}/{len(todo):,}", file=sys.stderr)
    fh.close()
    return cache


# =========================================================================== #
# 3. derive per-ORCID features (re-runnable without API)                        #
# =========================================================================== #
def orcid_features(cache):
    rows = []
    for orcid, d in cache.items():
        affs = d.get("affiliations", [])
        years = [a["start"] for a in affs if a["start"]]
        countries = [a["country"] for a in affs if a["country"]]
        # ordered career path by start year
        ordered = sorted([a for a in affs if a["start"]], key=lambda a: a["start"])
        country_seq = [a["country"] for a in ordered if a["country"]]
        rows.append({
            "orcid": orcid,
            "career_start": min(years) if years else None,     # academic-age anchor
            "n_affiliations": len(affs),
            "n_countries": len(set(countries)),
            "n_institutions": len(set(a["org"] for a in affs if a["org"])),
            "current_country": country_seq[-1] if country_seq else "",
            "first_country": country_seq[0] if country_seq else "",
            "ever_africa": any(c in AFRICA for c in countries),
            "ever_ssa": any(c in SSA for c in countries),
            "is_mobile": len(set(countries)) > 1,
            "country_path": ">".join(country_seq),             # e.g. "NG>GB>US"
        })
    f = pd.DataFrame(rows)
    print(f"[orcid] features for {len(f):,} researchers", file=sys.stderr)
    return f


# =========================================================================== #
# 4. attach to papers + answer the headline question                          #
# =========================================================================== #
def dominant_bloc(sb):
    parts = {x for x in S(sb).split(";") if x}
    for b in ("ssa", "maghreb"):
        if b in parts: return b
    return "other"

def attach(df, feat):
    fmap = feat.set_index("orcid").to_dict("index")

    def paper_row(row):
        orcids = [clean_orcid(o) for o in S(row.get("au_orcid_vec")).split("|")]
        orcids = [o for o in orcids if o]
        year = pd.to_numeric(row.get("oa_year"), errors="coerce")

        # first & last author ORCID (position-aligned to au_position_vec)
        pos = S(row.get("au_position_vec")).split("|")
        slots = S(row.get("au_orcid_vec")).split("|")
        first_o = last_o = ""
        for i, p in enumerate(pos):
            o = clean_orcid(slots[i]) if i < len(slots) else ""
            if p == "first" and o: first_o = o
            if p == "last" and o:  last_o = o

        def stage(o):
            cs = fmap.get(o, {}).get("career_start")
            if cs and pd.notna(year):
                return int(year) - cs
            return None

        senior_stage = stage(last_o)
        # returnee/diaspora signal: about-Africa paper, last author currently
        # NON-African, but PREVIOUSLY affiliated in Africa
        last_feat = fmap.get(last_o, {})
        study_bloc = S(row.get("study_bloc"))
        about_africa = ("ssa" in study_bloc) or ("maghreb" in study_bloc)
        prior_africa_now_out = bool(
            about_africa and last_feat and
            last_feat.get("ever_africa") and
            last_feat.get("current_country") not in AFRICA and
            last_feat.get("current_country"))

        return pd.Series({
            "first_author_career_yrs": stage(first_o),
            "last_author_career_yrs": senior_stage,
            "last_author_current_country": last_feat.get("current_country", ""),
            "last_author_is_mobile": last_feat.get("is_mobile", False),
            "last_author_country_path": last_feat.get("country_path", ""),
            "last_author_prior_africa_now_external": prior_africa_now_out,
            "n_authors_with_orcid": len(orcids),
        })

    add = df.apply(paper_row, axis=1)
    out = pd.concat([df, add], axis=1)
    out.to_parquet(OUT_PARQUET, index=False)
    print(f"[write] {OUT_PARQUET}", file=sys.stderr)

    # headline read-out for the question you asked

    if "study_bloc" not in out.columns:
        print("[validate] study_bloc absent — run on a parquet with study-location.",
              file=sys.stderr)
        return
    ab = out[out["study_bloc"].map(dominant_bloc)]

    ab = out[out["study_bloc"].fillna("").str.contains("ssa|maghreb")]
    if len(ab):
        share = ab["last_author_prior_africa_now_external"].mean()
        print(f"\n[readout] among about-Africa papers, senior authors now outside "
              f"Africa who PREVIOUSLY held an African affiliation: {share:.1%}",
              file=sys.stderr)
        print(f"[readout] ORCID coverage on senior authors: "
              f"{(out['last_author_career_yrs'].notna()).mean():.1%} of all papers",
              file=sys.stderr)
    return out

# =========================================================================== #
if __name__ == "__main__":
    df = pd.read_parquet(IN_PARQUET)
    # df = df_all[df_all["llm_label"].notna()]
    print(f"[in] {len(df):,} rows", file=sys.stderr)
    cache = harvest(all_orcids(df))
    feat = orcid_features(cache)
    feat.to_parquet(OUT_PARQUET.replace(".parquet", "_orcid_features.parquet"), index=False)
    attach(df, feat)


In [ ]:
feat

In [ ]:
cache

In [ ]:
m = attach(df, feat)
m